# IBGE favelas and urban communities, 2022

This is a field-name conversion of a preserved boundary file. The output `ibge_favelas_communities_2022.gpkg` retains all polygons and values. `datasets/run.py communities` executes these cells. [Codebook](CODEBOOK.md).

**Source:** the saved GeoPackage appears to derive from [IBGE's 2022 Favelas e Comunidades Urbanas product](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/tipologias-do-territorio/15788-favelas-e-comunidades-urbanas.html?lang=pt-BR). The original project supplied the file, without an acquisition/conversion script. Its inventory checksum defines the study input.

In [1]:
%run ../_shared.py
ctx = notebook_context("communities")
import pandas as pd
from IPython.display import display

## 1. Inspect and verify the source

The file contains 12,348 community polygons, one per community code, in EPSG:4674. Codes remain text identifiers; the file contains no resident counts. Source names and boundaries are preserved.

In [2]:
from processing import source_file
source = source_file(ctx.root)
baseline_checksum(ctx.root, source)
display(geopackage_inventory(source))
display(sql_table(source, "SELECT cd_fcu, nm_fcu, sigla_uf, cd_mun FROM favelas_2022 LIMIT 5"))

,Layer,Rows,Geometry,CRS
0,favelas_2022,12348,MultiPolygon,EPSG:4674


,cd_fcu,nm_fcu,sigla_uf,cd_mun
0,35503080069,Jardim Miliunas,SP,3550308
1,42024040016,Rua Araranguá,SC,4202404
2,33045570554,Tiqui,RJ,3304557
3,33045570104,Parque Nossa Senhora da Penha,RJ,3304557
4,35095020135,Núcleo Residencial Jardim das Andorinhas II,SP,3509502


## 2. Define the English field names

Only these seven names change. Community, municipality, and state names retain their original spelling. No cleaning, reprojection, splitting, or removal is applied. [Mapping and checked-copy implementation](../english_export.py).

In [3]:
from english_export import COMMUNITIES
display(pd.DataFrame(COMMUNITIES.items(), columns=["Source field", "English field"]))

,Source field,English field
0,cd_fcu,community_code
1,nm_fcu,community_name
2,cd_uf,state_code
3,nm_uf,state_name
4,sigla_uf,state_abbreviation
5,cd_mun,municipality_code
6,nm_mun,municipality_name


## 3. Write and compare the final file

The export edits a checked copy and records the mapping in `field_dictionary`. It compares every original value, feature ID, CRS, and geometry blob before publishing; the source remains intact.

In [4]:
from processing import export_geopackage, inspect_final
from english_export import verify_export
result = inspect_final(ctx.root, preview=True) if ctx.notebook_from_saved else export_geopackage(ctx.root)
verify_export(source, result["file"], "communities", baseline_checksum(ctx.root, source))
assert result["features"] == 12348
show_final(ctx, result)
display(sql_table(result["file"], "SELECT community_code, community_name, state_abbreviation, municipality_code FROM favelas_2022 LIMIT 5"))
print("All seven source values and all 12,348 geometries are preserved.")

,File,Layer,Features,CRS
0,datasets/communities/ibge_favelas_communities_...,favelas_2022,12348,EPSG:4674


SHA-256: 38faf2c44a93c5a522acf2840b4ae8517d7917651c116511fa7de06b18ed04b4


,community_code,community_name,state_abbreviation,municipality_code
0,35503080069,Jardim Miliunas,SP,3550308
1,42024040016,Rua Araranguá,SC,4202404
2,33045570554,Tiqui,RJ,3304557
3,33045570104,Parque Nossa Senhora da Penha,RJ,3304557
4,35095020135,Núcleo Residencial Jardim das Andorinhas II,SP,3509502


All seven source values and all 12,348 geometries are preserved.


**Limit:** the build documents the preserved file and its field conversion. It does not establish the file's exact historical download date or compare it with a newer IBGE release. The website's clipped/quantised tiles are a separate display output.

<details><summary>Execution verification</summary>

The full notebook command completed on 2026-10-27. All 12,348 records matched the preceding reviewed export in IDs, attributes, and geometry blobs. This is processing parity, rather than independent scientific validation.

</details>